# Data Storage — Cleaned Restaurant Dataset

Stores the cleaned dataset in a queryable **SQLite database** (primary) and exports a **Parquet** file (columnar format).

| Storage Layer | Format | Purpose |
|---|---|---|
| Primary | SQLite (`restaurants.db`) | Structured SQL queries |
| Secondary | Parquet (`california_restaurants_cleaned.parquet`) | Efficient columnar analysis |

In [ ]:
import pandas as pd
import sqlite3
import os

df = pd.read_csv('california_restaurants_cleaned.csv')
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
df.head(3)

## 1. SQLite Storage

Two tables:
- **`restaurants`** — core fields (name, location, price, rating, reviews)
- **`restaurant_categories`** — normalised category tags (one row per category per restaurant)

In [ ]:
DB_PATH = 'restaurants.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS restaurant_categories')
cursor.execute('DROP TABLE IF EXISTS restaurants')

cursor.execute('''
CREATE TABLE restaurants (
    id             INTEGER PRIMARY KEY AUTOINCREMENT,
    yelp_url       TEXT,
    name           TEXT NOT NULL,
    street_address TEXT NOT NULL,
    zip_code       TEXT NOT NULL,
    city           TEXT NOT NULL,
    state          TEXT NOT NULL,
    price_range    TEXT NOT NULL,
    phone          TEXT NOT NULL,
    rating         REAL NOT NULL,
    num_reviews    INTEGER NOT NULL,
    website        TEXT,
    menu_link      TEXT,
    review_outlier INTEGER NOT NULL DEFAULT 0
)
''')

cursor.execute('''
CREATE TABLE restaurant_categories (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    restaurant_id INTEGER NOT NULL REFERENCES restaurants(id),
    category      TEXT NOT NULL,
    rank          INTEGER NOT NULL
)
''')

conn.commit()
print('Tables created.')

In [ ]:
for _, row in df.iterrows():
    cursor.execute('''
        INSERT INTO restaurants
            (yelp_url, name, street_address, zip_code, city, state,
             price_range, phone, rating, num_reviews, website, menu_link, review_outlier)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
    ''', (
        row.get('Yelp URL'), row['Name'], row['Street Address'],
        row['Zip Code'], row['City'], row['State'],
        row['Price Range'], row['Phone'],
        float(row['Rating']), int(row['Number of Reviews']),
        row.get('Website'), row.get('Menu Link'),
        int(bool(row.get('review_outliers', False)))
    ))
    rest_id = cursor.lastrowid

    for rank, cat_col in enumerate(['Category 1', 'Category 2', 'Category 3'], start=1):
        cat = row.get(cat_col)
        if pd.notna(cat) and str(cat).strip():
            cursor.execute(
                'INSERT INTO restaurant_categories (restaurant_id, category, rank) VALUES (?,?,?)',
                (rest_id, str(cat).strip(), rank)
            )

conn.commit()

n_rest = cursor.execute('SELECT COUNT(*) FROM restaurants').fetchone()[0]
n_cats = cursor.execute('SELECT COUNT(*) FROM restaurant_categories').fetchone()[0]
print(f'Inserted {n_rest} restaurants and {n_cats} category records into {DB_PATH}')

## 2. Verify — SQL Queries

In [ ]:
# Average rating and review count by price range
q1 = pd.read_sql_query('''
    SELECT price_range,
           COUNT(*)                   AS num_restaurants,
           ROUND(AVG(rating), 2)      AS avg_rating,
           ROUND(AVG(num_reviews), 0) AS avg_reviews
    FROM restaurants
    GROUP BY price_range
    ORDER BY price_range
''', conn)
print('--- Average Rating by Price Range ---')
display(q1)

In [ ]:
# Top 10 cities by restaurant count
q2 = pd.read_sql_query('''
    SELECT city, COUNT(*) AS num_restaurants
    FROM restaurants
    GROUP BY city
    ORDER BY num_restaurants DESC
    LIMIT 10
''', conn)
print('--- Top 10 Cities ---')
display(q2)

In [ ]:
# Top 10 primary categories
q3 = pd.read_sql_query('''
    SELECT category, COUNT(*) AS count
    FROM restaurant_categories
    WHERE rank = 1
    GROUP BY category
    ORDER BY count DESC
    LIMIT 10
''', conn)
print('--- Top 10 Primary Categories ---')
display(q3)

conn.close()
print(f'\nDatabase file size: {os.path.getsize(DB_PATH) / 1024:.1f} KB')

## 3. Parquet Export

In [ ]:
PARQUET_PATH = 'california_restaurants_cleaned.parquet'
df.to_parquet(PARQUET_PATH, index=False)

df_verify = pd.read_parquet(PARQUET_PATH)
print(f'Parquet saved: {PARQUET_PATH}')
print(f'Shape: {df_verify.shape}')
print(f'Parquet size : {os.path.getsize(PARQUET_PATH) / 1024:.1f} KB')
print(f'CSV size     : {os.path.getsize("california_restaurants_cleaned.csv") / 1024:.1f} KB')